# Analysing Human Language using R — *talk* + *text* + *topics* workshop

✋ **NOTE** - You need to create a copy of this notebook before you work through it. This can be done by clicking on "Save a copy in Drive" option in the File menu.

<img src="https://r-talk.org/logo.png" alt="talk logo" width="200"> <img src="https://r-text.org/logo.png" alt="text logo" width="200"> <img src="https://r-topics.org/logo.png" alt="topics logo" width="200">

This notebook sets up **one shared Python environment** used by both the [talk](https://r-talk.org) package (audio: transcription, diarisation, speech embeddings) and the [text](https://r-text.org) package (text: language embeddings and analyses), together with the [topics](https://r-topics.org) package (topic modeling of language) — so you can go from a voice recording all the way to text-based analyses and topic models in a single session.


# 0. Setup — one click

Run the single cell below (click the ▶ play button on its left) and let it finish before continuing. It downloads a pre-built environment and takes roughly **5 minutes**. ☕

When it is done you will see **SETUP COMPLETE** at the bottom of the output, together with a test transcription and a test text embedding.

*Optional:* to use a GPU for faster models, first select **Runtime → Change runtime type → T4 GPU** (do this before running the setup cell, because changing runtime later erases the installation).


In [ ]:
## ════════════════════════════════════════════════════════════════
##  ONE-CLICK SETUP — click the play button on this cell, then wait.
##  Takes roughly ~5 minutes. Progress messages appear below.
## ════════════════════════════════════════════════════════════════
t0 <- Sys.time()

## ── Step 1/5: System tools ───────────────────────────────────────
# ffmpeg is required for transcription (whisper loads audio through the
# ffmpeg binary; it must come from apt, NOT conda). Java is needed by
# the text package (rJava).
cat("\n=== Step 1/5: Installing system tools (ffmpeg, Java) ===\n")
system("apt-get update -qq && apt-get install -y -qq ffmpeg pigz openjdk-11-jdk-headless")

## ── Step 2/5: Miniconda via condacolab ───────────────────────────
cat("\n=== Step 2/5: Installing Miniconda (condacolab) ===\n")
system("pip install -q condacolab gdown")
system("python - <<'PY'\nimport condacolab; condacolab.install()\nPY")

## ── Step 3/5: Download the pre-built environment ─────────────────
# One archive with everything: the shared talkrpp_condaenv conda
# environment (torch, WhisNemo, WhiSPA + the text-package Python
# stack), the talk/text/topics R packages, and all pre-downloaded
# models (whisper, NeMo diarisation, mxbai text embeddings).
cat("\n=== Step 3/5: Downloading pre-built environment (the long step) ===\n")
file_id <- "1d1n0HxYLOVvH4sptbFvjjE76MTyW5DYu"   # talk_text_topics_aug_2026.tar.gz
system(paste("gdown", file_id, "-O talk_text_topics_aug_2026.tar.gz"))
stopifnot(file.exists("talk_text_topics_aug_2026.tar.gz"))
cat("Unpacking…\n")
system("tar -I pigz -xf talk_text_topics_aug_2026.tar.gz -C /")
unlink("talk_text_topics_aug_2026.tar.gz")   # free disk space
.libPaths(c("/content/library", .libPaths()))

## ── Step 4/5: Wire Java into R (rJava, needed by text) ───────────
cat("\n=== Step 4/5: Configuring Java for R ===\n")
java_home <- dirname(dirname(system2("readlink", c("-f", Sys.which("javac")), stdout = TRUE)))
Sys.setenv(
  JAVA_HOME       = java_home,
  LD_LIBRARY_PATH = paste(file.path(java_home, "lib/server"),
                          Sys.getenv("LD_LIBRARY_PATH"), sep = ":")
)
system("R CMD javareconf")
dyn.load(file.path(java_home, "lib/server/libjvm.so"))
rjava_ok <- tryCatch({ library(rJava); TRUE }, error = function(e) {
  message("Pre-built rJava incompatible with this R — recompiling from source…")
  FALSE
})
if (!rjava_ok) {
  install.packages("rJava", repos = "https://cloud.r-project.org")
  library(rJava)
}
.jinit()   # 0 means the JVM loaded without errors

## ── Step 5/5: Initialize BOTH packages to the shared environment ─
cat("\n=== Step 5/5: Initializing talk and text (shared environment) ===\n")
library(reticulate)
Sys.setenv(RETICULATE_MINICONDA_PATH = system2("conda", c("info", "--base"), stdout = TRUE))
talk::talkrpp_initialize()
library(talk)
text::textrpp_initialize(condaenv = "talkrpp_condaenv", save_profile = FALSE)
library(text)
library(topics)

# Quick proof that everything works (all models are already in the
# archive, so this should only take seconds)
wav <- system.file("extdata/test_short.wav", package = "talk")
transcription <- talkText(wav)
print(transcription)
emb <- textEmbed(transcription$transcription,
                 model = "mixedbread-ai/mxbai-embed-large-v1")
cat("textEmbed dimensions:", dim(emb$texts[[1]]), "\n")

## ── Done ─────────────────────────────────────────────────────────
cat("\n============================================================\n")
cat("  SETUP COMPLETE in",
    round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1),
    "minutes — talk, text and topics are loaded and ready!\n")
cat("============================================================\n")


# 1. From voice to text

The `talk` package ships with a short example recording, so you can try it without uploading anything.


In [ ]:
# A short example recording bundled with the package
wav_path <- system.file("extdata/test_short.wav", package = "talk")
wav_path

# Speech-to-text transcription: a tibble with one row per file
transcription <- talkText(wav_path)
transcription


# 2. Speech embeddings (talk)

Embeddings represent the *sound* of the recording (voice acoustics, prosody) as numbers, ready for downstream analyses.


In [ ]:
embeddings_audio <- talkEmbed(wav_path)
dim(embeddings_audio)
embeddings_audio[, 1:5]


# 3. From voice to *language* analysis — talk + text together

This is where the shared environment pays off: transcribe speech with **talk**, then analyse the words with **text** — in the same R session. This uses the same embedding model as the workshop exercises.


In [ ]:
# Language embeddings of the transcribed words (text package)
embeddings_text <- textEmbed(
  transcription$transcription,
  model = "mixedbread-ai/mxbai-embed-large-v1"
)
embeddings_text$texts[[1]]


# 4. Who said what? (speaker diarisation)

For conversations, `talkTranscribeDiarise()` additionally separates the speakers. The example below uses a bundled two-speaker recording — this runs the full diarisation pipeline, so expect a few minutes on CPU (faster on GPU).


In [ ]:
wav_two_speakers <- system.file("extdata/test_diarise.wav", package = "talk")
conversation <- talkTranscribeDiarise(wav_two_speakers, num_speakers = 2)
conversation


# 5. Try your own recording

Upload a `.wav` file using the folder icon 📁 in the left sidebar (drag and drop), then point the functions at it, e.g.:

```r
my_transcript <- talkText("my_recording.wav")
my_embeddings <- textEmbed(my_transcript$transcription)
```

The `topics` package is also installed and loaded — see the tutorials on [r-topics.org](https://r-topics.org) for topic modeling of your transcribed language.
